# Comparative Legal Clause Classification

This notebook builds a reproducible NLP coursework pipeline for legal clause classification using LEDGAR as the main dataset. It compares dummy baselines, sparse classical NLP models, a fine-tuned transformer classifier, optional Qwen2.5-Instruct prompting, and a small human-in-the-loop review prototype.

The implementation now lives in `modules/`, matching the coursework-style layout used in the reference DLIA project. The notebook is kept as the orchestration and reporting layer.

## 1. Install, Imports, and Configuration

The flags below control expensive sections. Classical models run on CPU. Transformer and Qwen sections use GPU automatically when available and skip gracefully if the runtime is unsuitable.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys


def ensure_notebook_package(import_name: str, pip_name: str | None = None) -> None:
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])


for import_name, pip_name in [
    ("pandas", "pandas"),
    ("datasets", "datasets"),
    ("huggingface_hub", "huggingface_hub"),
    ("sklearn", "scikit-learn"),
    ("joblib", "joblib"),
    ("matplotlib", "matplotlib"),
]:
    ensure_notebook_package(import_name, pip_name)


def find_notebook_project_root(start: Path = Path.cwd()) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "modules").exists():
            return candidate
    return start


PROJECT_ROOT = find_notebook_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from IPython.display import display

from modules.data_setup import (
    adapt_cuad_to_clause_classification,
    build_project_paths,
    download_cuad_if_missing,
    load_cuad_raw_files,
    load_or_download_ledgar,
    print_dataset_availability,
    seed_everything,
)
from modules.preprocessing import create_ledgar_eda, preprocess_ledgar
from modules.baselines import run_baseline_experiments
from modules.classical_models import run_classical_experiments
from modules.transformer_model import train_transformer_classifier
from modules.qwen_prompting import run_qwen_baseline
from modules.agentic_review import run_agentic_review
from modules.evaluation import save_final_comparison
from modules.error_analysis import run_error_analysis

SEED = 42
DATASET_NAME = "LEDGAR"
TOP_K_LABELS = 20
MAX_FEATURES_LIST = [10000, 30000]
NGRAM_RANGES = [(1, 1), (1, 2)]
RUN_CLASSICAL_MODELS = True
RUN_TRANSFORMER = True
RUN_QWEN_BASELINE = True
RUN_AGENTIC_EXTENSION = True
RUN_NAIVE_BAYES = True
TRANSFORMER_MODEL_NAME = "distilbert-base-uncased"
OPTIONAL_LEGAL_MODEL_NAME = "nlpaueb/legal-bert-base-uncased"
QWEN_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
MAX_TRANSFORMER_LENGTH = 256
QWEN_EVAL_SAMPLE_SIZE = 200
QWEN_FEW_SHOT_EXAMPLES_PER_CLASS = 1

DOWNLOAD_LEDGAR_IF_MISSING = True
DOWNLOAD_CUAD_IF_MISSING = True
USE_HF_CACHE = True
FORCE_REDOWNLOAD = False

paths = build_project_paths(PROJECT_ROOT)
DEVICE = seed_everything(SEED)

print(f"Project root: {paths.project_root}")
print(f"Device: {DEVICE}")
try:
    import torch

    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("GPU is unavailable. Transformer/Qwen sections will skip or reduce work gracefully.")
except Exception:
    print("PyTorch is unavailable. Transformer/Qwen sections will skip if they require it.")

## 2. Dataset Download and Raw Setup

LEDGAR is downloaded with Hugging Face Datasets because it already has train, validation, and test splits for legal clause classification.

CUAD is downloaded separately because it is organised as a contract-review, question-answering, and span-extraction dataset. CUAD is not merged with LEDGAR; it can be adapted later by extracting annotated answer spans and assigning each span the corresponding CUAD category.

In [ ]:
ledgar_raw_splits = load_or_download_ledgar(
    paths,
    download_if_missing=DOWNLOAD_LEDGAR_IF_MISSING,
    force_redownload=FORCE_REDOWNLOAD,
)
cuad_json_path, master_clauses_path = download_cuad_if_missing(
    paths,
    download_if_missing=DOWNLOAD_CUAD_IF_MISSING,
    force_redownload=FORCE_REDOWNLOAD,
)
raw_cuad_json, master_clauses_df = load_cuad_raw_files(cuad_json_path, master_clauses_path)
cuad_clause_df = adapt_cuad_to_clause_classification(raw_cuad_json)
print_dataset_availability(ledgar_raw_splits, cuad_json_path, master_clauses_path, cuad_clause_df)

## 3. LEDGAR Preprocessing and EDA

The preprocessing standardises LEDGAR into a compact clause-classification schema, removes malformed examples and exact duplicate text-label pairs, selects the most frequent labels using only the training split, and preserves the official train/validation/test structure.

In [ ]:
processed_splits, label2id, id2label = preprocess_ledgar(
    ledgar_raw_splits,
    paths,
    top_k_labels=TOP_K_LABELS,
    dataset_name=DATASET_NAME,
)
split_summary = create_ledgar_eda(processed_splits, paths.results_dir)

if processed_splits:
    train_df = processed_splits["train"]
    validation_df = processed_splits["validation"]
    test_df = processed_splits["test"]
    label_names = [id2label[i] for i in sorted(id2label)]
    display(split_summary)
    display(pd.DataFrame({"label": label_names}))
else:
    train_df = validation_df = test_df = pd.DataFrame(columns=["text", "label", "label_id", "split", "source_dataset"])
    label_names = []
    print("Main LEDGAR experiment cannot run without LEDGAR data.")

## 4. Shared Result State

Each model section appends a result row and, when available, stores prediction tables for later error analysis.

In [ ]:
completed_results = []
prediction_tables = {}
trained_models = {}

## 5. Dummy Baselines

Random and majority baselines establish lower-bound performance before interpreting more complex supervised models.

In [ ]:
baseline_results, baseline_prediction_tables = run_baseline_experiments(
    train_df,
    test_df,
    id2label,
    paths.results_dir,
    dataset_name=DATASET_NAME,
    seed=SEED,
)
completed_results.extend(baseline_results)
prediction_tables.update(baseline_prediction_tables)
if baseline_results:
    display(pd.DataFrame(baseline_results)[["model_name", "accuracy", "macro_f1", "weighted_f1", "notes"]])

## 6. Classical TF-IDF Models

TF-IDF Logistic Regression, Linear SVM, and optional Multinomial Naive Bayes provide efficient sparse-text baselines. The validation split is used for model selection, then the selected configuration is evaluated once on the test split.

In [ ]:
best_classical_model = None
best_classical_name = None

if not RUN_CLASSICAL_MODELS:
    print("Classical models skipped because RUN_CLASSICAL_MODELS is False.")
else:
    classical_output = run_classical_experiments(
        train_df,
        validation_df,
        test_df,
        id2label,
        paths.results_dir,
        max_features_list=MAX_FEATURES_LIST,
        ngram_ranges=NGRAM_RANGES,
        dataset_name=DATASET_NAME,
        seed=SEED,
        run_naive_bayes=RUN_NAIVE_BAYES,
    )
    completed_results.extend(classical_output["results"])
    prediction_tables.update(classical_output["prediction_tables"])
    best_classical_model = classical_output["best_model"]
    best_classical_name = classical_output["best_model_name"]
    if best_classical_model is not None:
        trained_models["best_classical"] = best_classical_model
    if classical_output["results"]:
        display(pd.DataFrame(classical_output["results"])[["model_name", "accuracy", "macro_f1", "weighted_f1", "notes"]])

## 7. Fine-Tuned Transformer Classifier

DistilBERT is used as the preferred transformer because it is faster and practical for coursework hardware. LegalBERT can be substituted in the configuration if GPU resources allow. This section catches memory or environment failures and records a skip instead of stopping the notebook.

In [ ]:
transformer_output = train_transformer_classifier(
    train_df,
    validation_df,
    test_df,
    id2label,
    paths.results_dir,
    model_name=TRANSFORMER_MODEL_NAME,
    max_length=MAX_TRANSFORMER_LENGTH,
    dataset_name=DATASET_NAME,
    seed=SEED,
    run_transformer=RUN_TRANSFORMER,
)
if transformer_output["result"] is not None:
    completed_results.append(transformer_output["result"])
    prediction_tables[TRANSFORMER_MODEL_NAME] = transformer_output["predictions"]
    trained_models["transformer_trainer"] = transformer_output["trainer"]
    display(pd.DataFrame([transformer_output["result"]])[["model_name", "accuracy", "macro_f1", "weighted_f1"]])
elif transformer_output["skip_result"] is not None:
    completed_results.append(transformer_output["skip_result"])

## 8. Qwen2.5-Instruct Prompting Baseline

Qwen2.5-Instruct is used only as a zero-shot and few-shot prompting baseline. It is not fine-tuned. This tests whether an instruction-tuned causal language model can map legal clauses to allowed LEDGAR labels without task-specific training.

In [ ]:
qwen_output = run_qwen_baseline(
    train_df,
    test_df,
    label2id,
    id2label,
    paths.results_dir,
    model_name=QWEN_MODEL_NAME,
    label_names=label_names,
    eval_sample_size=QWEN_EVAL_SAMPLE_SIZE,
    few_shot_examples_per_class=QWEN_FEW_SHOT_EXAMPLES_PER_CLASS,
    dataset_name=DATASET_NAME,
    seed=SEED,
    run_qwen=RUN_QWEN_BASELINE,
)
completed_results.extend(qwen_output["results"])
qwen_predictions_df = qwen_output["predictions"]
qwen_invalid_outputs_df = qwen_output["invalid_outputs"]
qwen_model = qwen_output["model"]
qwen_tokenizer = qwen_output["tokenizer"]
if qwen_output["results"]:
    display(pd.DataFrame(qwen_output["results"])[["model_name", "accuracy", "macro_f1", "weighted_f1", "notes"]])

## 9. Small Agentic Review Prototype

This is a small human-in-the-loop demonstration inspired by ReAct/tool-use workflows. It is not a fully autonomous legal agent and does not provide legal advice. This output is for clause triage and research purposes only.

In [ ]:
agentic_examples_df = run_agentic_review(
    test_df,
    id2label,
    paths.results_dir,
    best_model=best_classical_model,
    qwen_model=qwen_model,
    qwen_tokenizer=qwen_tokenizer,
    run_agentic=RUN_AGENTIC_EXTENSION,
    seed=SEED,
)
if not agentic_examples_df.empty:
    display(agentic_examples_df.head(10))

## 10. Final Model Comparison

The final table records every completed or skipped model family without fabricating missing metrics. Prompted LLM results are based on the configured test sample size rather than the full test set.

In [ ]:
comparison_df = save_final_comparison(completed_results, paths.results_dir)
print(f"Saved final comparison to: {paths.results_dir / 'final_model_comparison.csv'}")
display(comparison_df)

## 11. Error Analysis

This section examines confused label pairs, misclassified examples, class imbalance, label ambiguity, long clauses, boilerplate wording, and Qwen invalid outputs where available.

In [ ]:
error_outputs = run_error_analysis(
    comparison_df,
    prediction_tables,
    train_df,
    paths.results_dir,
    best_classical_name=best_classical_name,
    transformer_model_name=TRANSFORMER_MODEL_NAME,
    qwen_predictions_df=qwen_predictions_df,
    qwen_invalid_outputs_df=qwen_invalid_outputs_df,
)

print(f"Best completed model by macro-F1: {error_outputs.get('best_model_name')}")
if "classical_confusions" in error_outputs:
    print("Top classical confused label pairs:")
    display(error_outputs["classical_confusions"])
if "classical_misclassified" in error_outputs:
    print("Classical misclassified examples:")
    display(error_outputs["classical_misclassified"][["text", "label", "predicted_label"]])
if "transformer_misclassified" in error_outputs:
    print("Transformer misclassified examples:")
    display(error_outputs["transformer_misclassified"][["text", "label", "predicted_label"]])
if "qwen_invalid_outputs" in error_outputs:
    print("Qwen invalid outputs:")
    display(error_outputs["qwen_invalid_outputs"].head(10))
if "qwen_plausible_nonmatching" in error_outputs:
    print("Qwen semantically plausible but non-matching label examples require manual inspection:")
    display(error_outputs["qwen_plausible_nonmatching"].head(10))

print("Class imbalance summary:")
display(error_outputs.get("class_imbalance", pd.DataFrame()).head(20))

## 12. Report-Ready Method Notes

**Aim.** Compare classical supervised models, fine-tuned transformer models, and instruction-tuned LLM prompting for LEDGAR legal clause classification.

**Dataset.** LEDGAR provides legal clause/provision texts with clause type labels and official train/validation/test splits. CUAD is downloaded separately because it has a span-extraction contract review format and is not merged into LEDGAR.

**Preprocessing.** The pipeline standardises columns, normalises whitespace only, removes empty examples and exact duplicate text-label pairs, selects the top-k labels using the training split, and preserves official splits.

**Baselines.** Random and majority baselines establish lower-bound performance for the selected label set.

**Classical models.** TF-IDF Logistic Regression and Linear SVM are efficient sparse-text baselines; Naive Bayes is included as an optional comparison.

**Transformer.** DistilBERT or LegalBERT tests whether contextual representations improve clause classification when fine-tuned on LEDGAR.

**Qwen.** Qwen2.5-Instruct is used as a zero-shot/few-shot prompting baseline to test whether an instruction-tuned LLM can classify clauses without task-specific fine-tuning.

**Agentic extension.** The prototype demonstrates a human-in-the-loop clause triage workflow using classifier confidence and optional LLM explanation. This output is for clause triage and research purposes only.

**Limitations.** LEDGAR labels are clause types, not legal risk labels. The models do not provide legal advice. Class imbalance affects macro-F1. Qwen prompting may be sensitive to prompt format. Comparing fine-tuned classifiers and prompted LLMs is not perfectly fair because their training/setup differs. The agentic workflow is illustrative and not production-ready.